# 🧴 Sistema de Recomendación de Perfumes con Machine Learning

Este notebook entrena y evalúa un modelo de clasificación que recomienda un perfume
a partir de 5 variables del usuario: **Edad**, **Clima**, **Ocasión**, **Presupuesto (Precio)**
y **Género** (0 = Hombre, 1 = Mujer).

**Contenido:**
1. Carga y exploración de datos (EDA)
2. Preparación de datos para el modelo
3. Comparación de múltiples algoritmos de clasificación
4. Optimización de hiperparámetros
5. Evaluación del modelo final (accuracy, precision, recall, F1, top-3 accuracy)
6. **Matriz de confusión**
7. Importancia de variables
8. **Múltiples pruebas con casos personalizados (nuevos usuarios)**


In [1]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [16]:
# Librerías principales
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Carga de datos

El dataset (`dataset_perfumes.csv`) contiene 1,300 registros y 50 perfumes distintos
(Nicho, Diseñador y Árabes), balanceados (26 registros por perfume).

In [4]:
df = pd.read_csv("DATASET_PROTOTIPO.csv")

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(10)


Filas: 9911  |  Columnas: 16


,Edad,Clima_Cod,Clima,Ocasion_Cod,Ocasion,Presupuesto_USD,Genero_Cod,Genero,Tipo_Perfume_Cod,Tipo_Perfume,Familia_Olfativa_Cod,Familia_Olfativa,Precio_Real_USD,Marca,Perfume_ID,Perfume_Recomendado
0,28,2,Templado,3,Formal/Noche,124.45,2,Femenino,1,Diseñador,9,Chypre,105,Giorgio Armani,68,Armani Si
1,49,5,Árido/Desértico,7,Evento especial/Boda,39.00,3,Unisex,3,Árabe,8,Especiada/Oud,32,Ard Al Zaafaran,169,Ameer Al Oudh
2,30,4,Húmedo,4,Fiesta/Salida nocturna,141.56,2,Femenino,1,Diseñador,6,Gourmand,115,Yves Saint Laurent,9,Black Opium
3,22,3,Frío,4,Fiesta/Salida nocturna,135.93,2,Femenino,1,Diseñador,6,Gourmand,95,Carolina Herrera,36,212 VIP
4,35,2,Templado,1,Diario/Casual,114.41,1,Masculino,1,Diseñador,1,Cítrica,100,Chanel,73,Chanel Allure Homme Sport
5,28,2,Templado,3,Formal/Noche,129.12,2,Femenino,1,Diseñador,4,Oriental/Ámbar,90,Jean Paul Gaultier,23,Classique
6,36,3,Frío,3,Formal/Noche,84.93,1,Masculino,1,Diseñador,5,Aromática/Fougère,65,Azzaro,26,Azzaro Pour Homme
7,39,3,Frío,3,Formal/Noche,348.62,1,Masculino,2,Nicho,8,Especiada/Oud,330,Amouage,230,Amouage Epic Man
8,45,3,Frío,6,Cita romántica,127.28,1,Masculino,1,Diseñador,8,Especiada/Oud,105,Givenchy,190,Gentleman Boisée
9,41,2,Templado,3,Formal/Noche,321.10,3,Unisex,2,Nicho,1,Cítrica,280,Xerjoff,107,Xerjoff Erba Pura


In [3]:
df.info()


NameError: name 'df' is not defined

In [ ]:
# Chequeo de valores nulos y duplicados
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()}")
print(f"\nPerfumes únicos: {df['Perfume_Recomendado'].nunique()}")


## 2. Análisis exploratorio (EDA)

Revisamos cómo se distribuyen las variables predictoras (Edad, Clima, Ocasión, Presupuesto)
y la variable objetivo (Tipo de perfume) antes de entrenar el modelo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Distribución de edad
axes[0, 0].hist(df["Edad"], bins=20, color="#6C5CE7", edgecolor="white")
axes[0, 0].set_title("Distribución de Edad")
axes[0, 0].set_xlabel("Edad")

# Distribución de presupuesto
axes[0, 1].hist(df["Presupuesto_USD"], bins=25, color="#00B894", edgecolor="white")
axes[0, 1].set_title("Distribución de Presupuesto (USD)")
axes[0, 1].set_xlabel("Presupuesto (USD)")

# Conteo por clima
sns.countplot(data=df, y="Clima", order=df["Clima"].value_counts().index,
              ax=axes[1, 0], hue="Clima", palette="crest", legend=False)
axes[1, 0].set_title("Registros por Clima")

# Conteo por ocasión
sns.countplot(data=df, y="Ocasion", order=df["Ocasion"].value_counts().index,
              ax=axes[1, 1], hue="Ocasion", palette="flare", legend=False)
axes[1, 1].set_title("Registros por Ocasión")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.countplot(data=df, x="Tipo_Perfume", order=["Diseñador", "Nicho", "Árabe"],
              ax=axes[0], hue="Tipo_Perfume", palette="Set2", legend=False)
axes[0].set_title("Registros por Tipo de Perfume")

sns.countplot(data=df, y="Familia_Olfativa",
              order=df["Familia_Olfativa"].value_counts().index,
              ax=axes[1], hue="Familia_Olfativa", palette="viridis", legend=False)
axes[1].set_title("Registros por Familia Olfativa")

plt.tight_layout()
plt.show()


In [ ]:
# Relación entre variables numéricas (codificadas) y correlación
corr_cols = ["Edad", "Clima_Cod", "Ocasion_Cod", "Presupuesto_USD",
             "Genero_Cod", "Tipo_Perfume_Cod", "Familia_Olfativa_Cod"]

plt.figure(figsize=(7, 5.5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación entre variables numéricas codificadas")
plt.tight_layout()
plt.show()


In [ ]:
# Presupuesto promedio por clima y por ocasión (para validar el patrón esperado)
plt.figure(figsize=(11, 4.5))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x="Clima", y="Presupuesto_USD", hue="Clima", palette="crest", legend=False)
plt.xticks(rotation=30, ha="right")
plt.title("Presupuesto por Clima")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="Edad", y="Presupuesto_USD".replace("Presupuesto_USD","Presupuesto_USD"))
plt.title("Edad vs Presupuesto (dispersión general)")

plt.tight_layout()
plt.show()


## 3. Preparación de datos para el modelo

Usamos 5 variables como *features* (**X**): `Edad`, `Clima_Cod`, `Ocasion_Cod`,
`Presupuesto_USD` y `Genero_Cod` (género del perfume/usuario: 1 = Masculino,
2 = Femenino, 3 = Unisex — más adelante, al pedir una recomendación, se parametriza
de forma simplificada como **0 = Hombre** y **1 = Mujer**).

La variable objetivo (**y**) es el perfume recomendado (`Perfume_Recomendado`, 50 clases).

Hacemos un split 80/20 **estratificado** (para mantener la proporción de cada perfume
en train y test) y dejamos aparte un 20% que el modelo nunca ve durante el entrenamiento.

In [5]:
FEATURES = ["Edad", "Clima_Cod", "Ocasion_Cod", "Presupuesto_USD", "Genero_Cod"]
TARGET = "Perfume_Recomendado"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Prueba (test):  {X_test.shape[0]} filas")
print(f"Número de clases (perfumes distintos): {y.nunique()}")


Entrenamiento: 7928 filas
Prueba (test):  1983 filas
Número de clases (perfumes distintos): 236


## 4. Prueba 1 — Comparación de múltiples algoritmos

Entrenamos 5 algoritmos distintos y comparamos su *accuracy* tanto con
**validación cruzada (5-fold)** sobre el set de entrenamiento como en el **set de prueba**.

In [13]:
modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Regresión Logística": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados = []

for nombre, modelo in modelos.items():
    cv_scores = cross_val_score(modelo, X_train, y_train, cv=skf, scoring="accuracy")
    modelo.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, modelo.predict(X_test))
    resultados.append({
        "Modelo": nombre,
        "CV Accuracy (media)": cv_scores.mean(),
        "CV Accuracy (std)": cv_scores.std(),
        "Test Accuracy": test_acc,
    })

df_resultados = pd.DataFrame(resultados).sort_values("Test Accuracy", ascending=False)
df_resultados.reset_index(drop=True)


C:\Users\felip\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\felip\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://s

,Modelo,CV Accuracy (media),CV Accuracy (std),Test Accuracy
0,Random Forest,0.323286,0.006117,0.339385
1,Árbol de Decisión,0.299321,0.008298,0.315179
2,Naive Bayes,0.261353,0.004977,0.272315
3,Regresión Logística,0.195763,0.010067,0.229955
4,KNN (k=5),0.079213,0.005606,0.093797


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(df_resultados))

ax.bar(x_pos - 0.2, df_resultados["CV Accuracy (media)"], width=0.4,
       label="CV Accuracy (train, 5-fold)", color="#0984E3")
ax.bar(x_pos + 0.2, df_resultados["Test Accuracy"], width=0.4,
       label="Test Accuracy", color="#E17055")

ax.set_xticks(x_pos)
ax.set_xticklabels(df_resultados["Modelo"], rotation=20, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Comparación de algoritmos: CV vs Test Accuracy")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Prueba 2 — Optimización de hiperparámetros (GridSearchCV)

Tomamos el mejor algoritmo de la prueba anterior (normalmente **Random Forest**) y
buscamos su mejor combinación de hiperparámetros con validación cruzada.

In [7]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 8, 12, 16],
    "min_samples_split": [2, 4],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor accuracy (CV): {grid.best_score_:.4f}")

modelo_final = grid.best_estimator_


C:\Users\felip\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
21 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\felip\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\felip\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\felip\AppDat

Mejores hiperparámetros: {'max_depth': 12, 'min_samples_split': 4, 'n_estimators': 300}
Mejor accuracy (CV): 0.3344


## 6. Evaluación del modelo final

Evaluamos el modelo final sobre el set de prueba (datos nunca vistos) con varias métricas:
**accuracy**, **precision**, **recall**, **F1-score** y **top-3 accuracy**
(si el perfume correcto está entre las 3 mejores predicciones del modelo).

In [8]:
y_pred = modelo_final.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")


Accuracy:  0.3671
Precision: 0.4052
Recall:    0.3671
F1-score:  0.3597


In [9]:
# Top-3 accuracy: ¿el perfume correcto está entre las 3 mejores probabilidades predichas?
proba = modelo_final.predict_proba(X_test)
clases = modelo_final.classes_

top3_hits = 0
for i, real in enumerate(y_test.values):
    top3_idx = np.argsort(proba[i])[-3:]
    top3_perfumes = clases[top3_idx]
    if real in top3_perfumes:
        top3_hits += 1

top3_accuracy = top3_hits / len(y_test)
print(f"Top-3 Accuracy: {top3_accuracy:.4f}  "
      f"({top3_hits}/{len(y_test)} veces el perfume correcto estuvo entre las 3 mejores sugerencias)")


Top-3 Accuracy: 0.6369  (1263/1983 veces el perfume correcto estuvo entre las 3 mejores sugerencias)


In [10]:
print(classification_report(y_test, y_pred, zero_division=0))


                                  precision    recall  f1-score   support

                         212 Men       0.00      0.00      0.00         8
                         212 VIP       0.31      0.89      0.46         9
                    Acqua di Giò       1.00      0.50      0.67         8
                       Afnan 9pm       1.00      1.00      1.00         9
                 Afnan Supremacy       0.60      0.75      0.67         8
                   Ajmal Nafisah       0.57      0.50      0.53         8
                   Ajmal Shukran       0.29      0.50      0.36         8
                     Ajmal Wisal       1.00      0.78      0.88         9
                           Alien       0.00      0.00      0.00         9
                      Amber Musk       1.00      1.00      1.00         8
          Amber Oud Gold Edition       0.12      0.12      0.12         8
                   Ameer Al Oudh       0.50      0.38      0.43         8
                 Ameerat Al Arab     

## 7. Matriz de confusión

Como son 50 clases (perfumes), mostramos la matriz completa como *heatmap* y además una
versión **normalizada por fila** (porcentaje de aciertos por cada perfume real), que es más
fácil de leer que los conteos absolutos.

In [ ]:
labels_orden = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_orden)

plt.figure(figsize=(18, 15))
sns.heatmap(cm, xticklabels=labels_orden, yticklabels=labels_orden,
            cmap="Blues", cbar=True, square=True, linewidths=0.2, linecolor="lightgray")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.xlabel("Predicción del modelo")
plt.ylabel("Perfume real")
plt.title("Matriz de Confusión — Modelo final (50 clases)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Versión normalizada (proporción por fila) -> diagonal = % de aciertos por perfume
cm_norm = confusion_matrix(y_test, y_pred, labels=labels_orden, normalize="true")

plt.figure(figsize=(18, 15))
sns.heatmap(cm_norm, xticklabels=labels_orden, yticklabels=labels_orden,
            cmap="YlGnBu", cbar=True, square=True, vmin=0, vmax=1,
            linewidths=0.2, linecolor="lightgray")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.xlabel("Predicción del modelo")
plt.ylabel("Perfume real")
plt.title("Matriz de Confusión Normalizada (proporción por fila)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Aciertos en la diagonal: perfumes que el modelo predice mejor / peor
diagonal = pd.Series(np.diag(cm_norm), index=labels_orden).sort_values()

fig, ax = plt.subplots(figsize=(9, 12))
diagonal.plot(kind="barh", color=np.where(diagonal.values < 0.5, "#E17055", "#00B894"), ax=ax)
ax.set_xlabel("Proporción de aciertos (recall por clase)")
ax.set_title("Perfumes mejor vs peor identificados por el modelo")
plt.tight_layout()
plt.show()


## 8. Importancia de las variables

¿Cuáles de las 5 variables (Edad, Clima, Ocasión, Presupuesto, Género) pesan más al momento
de que el modelo elija un perfume?

In [ ]:
importancias = pd.Series(modelo_final.feature_importances_, index=FEATURES).sort_values()

plt.figure(figsize=(7, 4))
importancias.plot(kind="barh", color="#6C5CE7")
plt.title("Importancia de cada variable en el modelo final")
plt.xlabel("Importancia relativa")
plt.tight_layout()
plt.show()

importancias.sort_values(ascending=False)


## 9. Múltiples pruebas con casos personalizados (nuevos usuarios)

Probamos el modelo con perfiles de usuario inventados —que no están en el dataset—
para ver qué recomienda en escenarios realistas. Usamos `predict_proba` para mostrar
las **3 mejores recomendaciones** en cada caso, no solo la primera.

In [11]:
# Mapas de codificación (deben coincidir con los usados al construir el dataset)
CLIMA_COD = {
    "Cálido/Tropical": 1, "Templado": 2, "Frío": 3,
    "Húmedo": 4, "Árido/Desértico": 5,
}
OCASION_COD = {
    "Diario/Casual": 1, "Oficina/Trabajo": 2, "Formal/Noche": 3,
    "Fiesta/Salida nocturna": 4, "Deportivo/Ejercicio": 5,
    "Cita romántica": 6, "Evento especial/Boda": 7,
}
# Género simplificado para pedir la recomendación: 0 = Hombre, 1 = Mujer.
# Se traduce internamente a la codificación del dataset (Genero_Cod: 1=Masculino, 2=Femenino).
GENERO_COD = {0: 1, 1: 2}

def recomendar_perfume(edad, clima, ocasion, presupuesto, genero, top_n=3, modelo=modelo_final):
    """Recomienda los `top_n` perfumes más probables para un perfil de usuario.

    genero: 0 = Hombre, 1 = Mujer
    """
    if genero not in GENERO_COD:
        raise ValueError("genero debe ser 0 (Hombre) o 1 (Mujer)")

    fila = pd.DataFrame([{
        "Edad": edad,
        "Clima_Cod": CLIMA_COD[clima],
        "Ocasion_Cod": OCASION_COD[ocasion],
        "Presupuesto_USD": presupuesto,
        "Genero_Cod": GENERO_COD[genero],
    }])
    proba = modelo.predict_proba(fila)[0]
    clases = modelo.classes_
    top_idx = np.argsort(proba)[::-1][:top_n]
    return [(clases[i], round(proba[i] * 100, 1)) for i in top_idx]


In [12]:
casos_de_prueba = [
    dict(nombre="Hombre joven, verano, uso diario, presupuesto bajo",
         edad=22, clima="Cálido/Tropical", ocasion="Diario/Casual", presupuesto=80, genero=0),
    dict(nombre="Mujer adulta, invierno, evento formal, presupuesto alto",
         edad=45, clima="Frío", ocasion="Formal/Noche", presupuesto=300, genero=1),
    dict(nombre="Hombre 30 años, clima árido, boda, presupuesto medio",
         edad=30, clima="Árido/Desértico", ocasion="Evento especial/Boda", presupuesto=65, genero=0),
    dict(nombre="Mujer joven adulta, clima templado, oficina, presupuesto medio",
         edad=27, clima="Templado", ocasion="Oficina/Trabajo", presupuesto=100, genero=1),
    dict(nombre="Hombre adulto mayor, clima frío, cita romántica, presupuesto alto",
         edad=52, clima="Frío", ocasion="Cita romántica", presupuesto=280, genero=0),
    dict(nombre="Mujer joven, clima húmedo, fiesta nocturna, presupuesto bajo",
         edad=20, clima="Húmedo", ocasion="Fiesta/Salida nocturna", presupuesto=35, genero=1),
]

for caso in casos_de_prueba:
    genero_txt = "Hombre" if caso["genero"] == 0 else "Mujer"
    print(f"👤 {caso['nombre']}")
    print(f"   Edad={caso['edad']} | Clima={caso['clima']} | "
          f"Ocasión={caso['ocasion']} | Presupuesto=${caso['presupuesto']} | Género={genero_txt}")
    recomendaciones = recomendar_perfume(
        caso["edad"], caso["clima"], caso["ocasion"], caso["presupuesto"], caso["genero"]
    )
    for i, (perfume, prob) in enumerate(recomendaciones, start=1):
        print(f"   {i}. {perfume}  ({prob}% de confianza)")
    print("-" * 70)


👤 Hombre joven, verano, uso diario, presupuesto bajo
   Edad=22 | Clima=Cálido/Tropical | Ocasión=Diario/Casual | Presupuesto=$80 | Género=Hombre
   1. Invictus  (21.8% de confianza)
   2. Lacoste L.12.12 Blanc  (17.9% de confianza)
   3. L'Eau d'Issey Pour Homme  (6.7% de confianza)
----------------------------------------------------------------------
👤 Mujer adulta, invierno, evento formal, presupuesto alto
   Edad=45 | Clima=Frío | Ocasión=Formal/Noche | Presupuesto=$300 | Género=Mujer
   1. Portrait of a Lady  (30.8% de confianza)
   2. More Than Words  (21.1% de confianza)
   3. Love Don't Be Shy  (17.0% de confianza)
----------------------------------------------------------------------
👤 Hombre 30 años, clima árido, boda, presupuesto medio
   Edad=30 | Clima=Árido/Desértico | Ocasión=Evento especial/Boda | Presupuesto=$65 | Género=Hombre
   1. Oudh Ameer  (49.9% de confianza)
   2. Sheikh Al Fakhamah  (30.3% de confianza)
   3. Afnan Supremacy  (8.1% de confianza)
-------------

## 9.1 Uso rápido — solo parámetros

Si ya tienes el modelo entrenado (`modelo_final`) y solo quieres probar perfiles
nuevos sin escribir texto descriptivo ni imprimir nada extra, basta con llamar
directamente a `recomendar_perfume(edad, clima, ocasion, presupuesto, genero)`.

Al ser la última línea de la celda, Jupyter muestra el resultado automáticamente
(lista de tuplas `(perfume, % de confianza)`). Solo cambia los 5 valores.

Valores válidos:
- `clima`: "Cálido/Tropical", "Templado", "Frío", "Húmedo", "Árido/Desértico"
- `ocasion`: "Diario/Casual", "Oficina/Trabajo", "Formal/Noche", "Fiesta/Salida nocturna", "Deportivo/Ejercicio", "Cita romántica", "Evento especial/Boda"
- `genero`: **0 = Hombre**, **1 = Mujer**

In [28]:
recomendar_perfume(19, "Frío", "Oficina/Trabajo", 150, 0)

[('Sauvage EDP', np.float64(53.6)),
 ('Y EDP', np.float64(15.9)),
 ("Terre d'Hermès", np.float64(4.5))]

In [ ]:
recomendar_perfume(19, "Frío", "Cita romántica", 400, 0)

## 10. Conclusiones

- Se compararon **5 algoritmos** de clasificación (Random Forest, Árbol de Decisión, KNN,
  Regresión Logística y Naive Bayes) usando validación cruzada y set de prueba.
- El modelo final se optimizó con **GridSearchCV** y se evaluó con múltiples métricas
  (accuracy, precision, recall, F1 y top-3 accuracy), además de la **matriz de confusión**
  (absoluta y normalizada) para ver en qué perfumes el modelo acierta o se confunde más.
- Se probó el modelo con **6 perfiles de usuario nuevos** (no vistos en el entrenamiento),
  devolviendo en cada caso las 3 recomendaciones más probables con su nivel de confianza.
- Con solo 26 ejemplos por perfume, el modelo funciona como una prueba de concepto:
  para producción se recomienda ampliar el dataset con más variedad de perfiles por perfume.


In [ ]:
recomendar_perfume(19, "Frío", "Diario/Casual", 300, 0)